# Workshop 9. Logit and probit for a yes/no outcome (solutions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture09/workshop09_solutions.ipynb)

In this workshop you write the code yourselves. Work in your group or alone, and ask the TAs when you are stuck.

Task 1 is the lecture's walkthrough in code, fully worked: run it, answer the questions and change one thing. Tasks 2
to 4 start from an empty cell: predicted churn across visits for app and web users, a likelihood ratio test for
region, and the LPM, logit and probit side by side with your choice between them. The extension tasks are for those
who have time left. Exercises in the style of the textbook, with small numbers and no data, are in the TA's separate
notebook for this session (link on Moodle).

Each core task has two hints: the first names the statistical step, the second names the functions. Try the task
before you open them. The toolbox says where you have seen each function before, so you can look it up in that
notebook. The code patterns, the slides and the lecture notebook are your main references. You are welcome to use an
LLM, but every task can be solved without one, and you should be able to explain the statistics in your answers
yourself.

In every task, say what y and the x's are before you fit anything. Read the sign and significance from the
coefficients, and the sizes as probabilities or percentage points. Each answer is in words a manager can follow, with
the numbers it rests on.

## Toolbox

The functions you need today, and where you have seen them in use.

| Function | What it does | Seen in |
|---|---|---|
| `smf.logit("y ~ x1 + x2", data=df).fit()` | Fits a logit model by maximum likelihood; y is 0 or 1 | Lecture 9, Part 2 |
| `smf.probit(...)` | The same with the normal curve | Lecture 9, Part 4 |
| `smf.ols(...).fit(cov_type="HC1")` | The linear probability model with robust standard errors | Lecture 8, Part 4 |
| `model.summary()`, `model.params`, `model.pvalues` | The output, the coefficients, the p-values | Lecture 6; Lecture 9, Part 2 |
| `model.llf`, `model.prsquared` | The log-likelihood and McFadden's pseudo R² | Lecture 9, Part 2 |
| `stats.chi2.sf(LR, q)` | The p-value of a likelihood ratio test with q degrees of freedom | Lecture 9, Part 2 (Exercise 2) |
| `model.predict(pd.DataFrame({...}))` | Predicted probabilities; one column for every x in the formula | Lecture 8, Part 4; Lecture 9, Part 3 |
| `model.get_margeff(at="overall", dummy=True)` | Average marginal effects; `.summary()` or `.summary_frame()` | Lecture 9, Part 4 |
| `model.fittedvalues` | ŷ, one per row; in an LPM the predicted probabilities | Lecture 8, Part 4 |
| `np.arange(0, 91, 5)` | The numbers 0, 5, ..., 90 | Lecture 9, Part 3 (the gap figure) |
| `df[df["x"].isin([10, 30, 60])]`, `df.sort_values("x", ascending=False).head(200)` | Rows with given values of x; the 200 rows with the largest x | New today: see the first code pattern |

## Code patterns

Two complete patterns for today. Replace `df`, `y01`, `d1`, `g`, `x` and the values with your data and column names.
The patterns are templates; they do not run as they stand.

**A logit model, predictions for described cases and across a grid, and average marginal effects**

```python
logit = smf.logit("y01 ~ d1 + C(g) + x", data=df).fit()
print(logit.summary())                     # signs, z, p; converged; log-likelihood; pseudo R2; LLR p-value

new = pd.DataFrame({"d1": [0, 1], "g": "A", "x": 3})       # the same case with d1 = 0 and d1 = 1
p = logit.predict(new)
print(p.round(3))
print(f"Gap: {100 * (p.iloc[0] - p.iloc[1]):.1f} percentage points")      # d1 = 0 minus d1 = 1

grid = pd.DataFrame({"d1": 0, "g": "A", "x": np.arange(0, 91, 5)})   # one row per value of x
grid["p"] = logit.predict(grid)
plt.plot(grid["x"], grid["p"])
print(grid[grid["x"].isin([10, 30, 60])])                          # only the rows for these values of x

df["p"] = logit.predict(df)
top = df.sort_values("p", ascending=False).head(200)              # the 200 rows with the highest p

ame = logit.get_margeff(at="overall", dummy=True)
print(ame.summary())                       # dy/dx times 100: percentage points
```

**A likelihood ratio test for a group of x's, and three models side by side**

```python
small = smf.logit("y01 ~ d1 + x", data=df).fit(disp=0)
large = smf.logit("y01 ~ d1 + x + C(g)", data=df).fit(disp=0)          # the same rows, more x's
LR = 2 * (large.llf - small.llf)
q = len(large.params) - len(small.params)                              # number of x's added
print(f"LR = {LR:.2f}, df = {q}, p = {stats.chi2.sf(LR, q):.3f}")

formula = "y01 ~ d1 + C(g) + x"
lpm = smf.ols(formula, data=df).fit(cov_type="HC1")
logit = smf.logit(formula, data=df).fit(disp=0)
probit = smf.probit(formula, data=df).fit(disp=0)
table = pd.DataFrame({
    "LPM": lpm.params.drop("Intercept"),
    "logit AME": logit.get_margeff(at="overall", dummy=True).margeff,
    "probit AME": probit.get_margeff(at="overall", dummy=True).margeff})
print((100 * table).round(2))                                          # percentage points
```

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
customers = pd.read_csv(url + "customers_sample.csv")   # 1,000 RetailCorp customers drawn at random
customers.head()

## Core tasks

### Task 1. Customer A and the average gap for members (worked)

The lecture's walkthrough in code. Run the cells and answer the questions below them.

y = `churned` (1 if no order in January to June 2026); the x's are `member`, `channel`, `tenure_years`, `age` and
`visits_2025`, as in the Lecture 8 LPM.

In [ ]:
m_logit = smf.logit("churned ~ member + C(channel) + tenure_years + age + visits_2025", data=customers).fit()
print(m_logit.summary())

In [ ]:
channel = "App"      # change to "Web"

customer_a = pd.DataFrame({"member": [0, 1], "channel": channel, "tenure_years": 1, "age": 45, "visits_2025": 15})
p = m_logit.predict(customer_a)
print(f"Not a member: {p.iloc[0]:.3f}")
print(f"Member:       {p.iloc[1]:.3f}")
print(f"Gap: {100 * (p.iloc[0] - p.iloc[1]):.1f} percentage points")

In [ ]:
ame = m_logit.get_margeff(at="overall", dummy=True)
print(ame.summary())

**Questions**

1. What do the signs and p-values of the coefficients tell you? Why do we not read 0.4932 on Web as a change in
   probability?
2. What are the log-likelihood, LL-Null and the pseudo R², and what does the LLR p-value test?
3. Read the predicted probabilities for customer A, and the average marginal effect of membership with its interval.
   Why is the gap for customer A larger than the average marginal effect?

**Change one thing.** In the second cell, change `channel` to `"Web"` and run it. How do the two probabilities and
the gap change, and why?

**Answer.**
1. At the same values of the other x's, web customers churn more often than app users (z = 2.91, p = 0.004), and
   members (p < 0.001), customers with longer tenure (p < 0.001) and customers with more visits (p < 0.001) churn less
   often. Age shows no clear association (p = 0.23). The coefficients are on the log-odds scale: 0.4932 moves the
   index z, and how much that moves the probability depends on where the customer is on the curve. So we read the
   sign and the significance, and the sizes as probabilities.
2. The log-likelihood, −537.38, measures the fit of this model; closer to 0 is better. LL-Null, −602.15, is the fit
   with only an intercept, which gives every customer 29.0%. The pseudo R² is 1 − (−537.38 / −602.15) = 0.108. The
   LLR p-value, 3.0e-26, tests that all five slopes are 0, and that is clearly rejected.
3. Customer A, a new app user with 15 visits, has a predicted probability of 48.0% of churning as a non-member and
   33.5% as a member, 14.5 points lower. On average over the 1,000 customers, members are 10.8 points less likely to
   churn (95% interval 5.1 to 16.5) at the same channel, tenure, age and visits. Customer A is near the steep middle
   of the curve, where a change in z moves the probability most. Most customers have lower predicted probabilities
   than A (the share who churned is 29.0%), where the curve is flatter, so their gaps are smaller.

**As a web user.** 60.2% as a non-member and 45.2% as a member, a gap of 15.0 points. Web customers churn more
often, which moves customer A to a probability close to 0.5, where the curve is steepest, so the same member
coefficient gives a slightly larger gap.

### Task 2. Predicted churn across visits for app and web users

The marketing team wants to see how the probability of churn falls with visits, for app and for web users. Use
`m_logit` from Task 1.

1. Make a table of predicted probabilities for non-members with 3 years as customers, aged 45, for visits from 0 to
   90 in steps of 5, once for app users and once for web users. Plot the two curves in one figure.
2. Read the difference between web and app users, in percentage points, at 10, 30 and 60 visits.
3. Why does the difference change with visits, when the model has one coefficient for Web?

<details>
<summary>Hint 1</summary>

A predicted probability is the model's answer for a customer you describe. Describe the same customer at many values
of visits, once with channel App and once with channel Web, and the curves show how the probability changes. In a
logit model the same change in z moves the probability most near 0.5.
</details>

<details>
<summary>Hint 2</summary>

The first code pattern: `pd.DataFrame` with `"visits_2025": np.arange(0, 91, 5)` and the other columns fixed, then
`m_logit.predict(grid)`. Do it once with `"channel": "App"` and once with `"channel": "Web"`. Print the rows for 10,
30 and 60 visits with `.isin([10, 30, 60])`, as in the first code pattern.
</details>

In [ ]:
visits = np.arange(0, 91, 5)
curves = {}
for ch in ["App", "Web"]:
    grid = pd.DataFrame({"member": 0, "channel": ch, "tenure_years": 3, "age": 45, "visits_2025": visits})
    curves[ch] = m_logit.predict(grid).values
    plt.plot(visits, curves[ch], label=ch)
plt.xlabel("Visits in 2025")
plt.ylabel("Predicted probability of churn")
plt.legend()
plt.show()

In [ ]:
table = pd.DataFrame({"visits": visits, "App": curves["App"], "Web": curves["Web"]})
table["difference, points"] = 100 * (table["Web"] - table["App"])
print(table[table["visits"].isin([10, 30, 60])].round(3))

1. Both curves fall with visits and flatten out as they approach 0. The web curve lies above the app curve.
2. For a non-member with 3 years, aged 45, web users have a higher predicted probability of churn than app users:
   57.3% against 45.0% at 10 visits (12.3 points), 38.1% against 27.3% at 30 visits (10.8 points), and 16.1% against
   10.5% at 60 visits (5.6 points).
3. The Web coefficient, 0.4932, moves z by the same amount at every number of visits. At 10 visits the customer is
   near the steep middle of the curve, so that move changes the probability a lot. At 60 visits the probabilities are
   close to 0, where the curve is flat, so the same move changes the probability less. In an LPM the difference would
   be 10.1 points at any number of visits.

### Task 3. Does region help to predict churn?

1. Add `C(region)` to the model of Task 1. How many coefficients does that add, and which region is the reference?
2. Use a likelihood ratio test to answer whether the region dummies together help to predict churn, given the other
   x's. Give LR, the degrees of freedom and the p-value.
3. Compute the average marginal effects of the larger model. What do they say about the regions, and does membership
   change?

<details>
<summary>Hint 1</summary>

The likelihood ratio test compares the log-likelihoods of two models fitted on the same customers: the larger model
with region and the smaller one without. LR = 2 (ℓ larger − ℓ smaller) has a χ² distribution with as many degrees of
freedom as the number of coefficients added.
</details>

<details>
<summary>Hint 2</summary>

The second code pattern: `m_region = smf.logit(... + C(region) ..., data=customers).fit()`, then
`2 * (m_region.llf - m_logit.llf)` and `stats.chi2.sf(LR, 4)`. For the AMEs, `get_margeff(at="overall", dummy=True)`.
</details>

In [ ]:
m_region = smf.logit("churned ~ member + C(channel) + tenure_years + age + visits_2025 + C(region)",
                     data=customers).fit()
print(m_region.params.round(4))

In [ ]:
LR = 2 * (m_region.llf - m_logit.llf)
q = len(m_region.params) - len(m_logit.params)
print(f"Log-likelihood {m_logit.llf:.2f} without region, {m_region.llf:.2f} with region")
print(f"LR = {LR:.2f}, df = {q}, p = {stats.chi2.sf(LR, q):.3f}")

In [ ]:
ame_region = m_region.get_margeff(at="overall", dummy=True).summary_frame()
print(ame_region[["dy/dx", "Pr(>|z|)"]].round(4))

1. Four coefficients, one for each region except the reference, Hovedstaden (the first in alphabetical order).
2. LR = 2 × (−534.81 − (−537.38)) = 5.14 with 4 degrees of freedom, p = 0.27. There is no clear evidence that
   region helps to predict churn once membership, channel, tenure, age and visits are in the model.
3. Compared with Hovedstaden, the regions differ by −0.9 (Midtjylland), +9.5 (Nordjylland), +4.3 (Sjælland) and +1.7
   (Syddanmark) percentage points, with p-values from 0.06 to 0.82. Nordjylland's p = 0.06 is above 5%, and with four
   comparisons one p-value this low is not unusual; the joint test does not reject. The average
   marginal effect of membership is −10.6 points, almost the same as without region (−10.8), so leaving region out
   does not change the answer about members.

### Task 4. The LPM, logit and probit side by side

Fit the model of Task 1 as an LPM (with robust standard errors), a logit and a probit model.

1. How many predicted probabilities are below 0 or above 1 in each model?
2. Put the LPM coefficients and the logit and probit average marginal effects side by side, in percentage points. How
   close are they?
3. Predict the probability of churn in all three models for two customers: (C) a web customer who is not a member,
   aged 30, with half a year as a customer and 8 visits; (D) an app user who is a member, aged 60, with 9 years and 70
   visits.
4. In three or four sentences: which model would you report to RetailCorp's management, and why?

<details>
<summary>Hint 1</summary>

The three models answer the same question with different curves: a straight line, the logistic curve and the normal
curve. Compare them where it matters: the averages, and the predictions for customers far from the middle of the
data.
</details>

<details>
<summary>Hint 2</summary>

The second code pattern for the three models and the table. `model.fittedvalues` for the LPM and
`model.predict(customers)` for logit and probit give one predicted probability per customer; count the values below 0
with `(p < 0).sum()`. For the two customers, one `pd.DataFrame` with two rows and `model.predict(new)` in each model.
</details>

In [ ]:
formula = "churned ~ member + C(channel) + tenure_years + age + visits_2025"
lpm = smf.ols(formula, data=customers).fit(cov_type="HC1")
logit = smf.logit(formula, data=customers).fit(disp=0)
probit = smf.probit(formula, data=customers).fit(disp=0)

for name, p in [("LPM", lpm.fittedvalues), ("logit", logit.predict(customers)), ("probit", probit.predict(customers))]:
    print(f"{name:7s} below 0: {(p < 0).sum():3d}, above 1: {(p > 1).sum()}, lowest {p.min():.3f}, highest {p.max():.3f}")

In [ ]:
table = pd.DataFrame({
    "LPM": lpm.params.drop("Intercept"),
    "logit AME": logit.get_margeff(at="overall", dummy=True).margeff,
    "probit AME": probit.get_margeff(at="overall", dummy=True).margeff})
print((100 * table).round(2))

In [ ]:
new = pd.DataFrame({"member": [0, 1], "channel": ["Web", "App"], "tenure_years": [0.5, 9],
                    "age": [30, 60], "visits_2025": [8, 70]}, index=["C", "D"])
print(pd.DataFrame({"LPM": lpm.predict(new), "logit": logit.predict(new), "probit": probit.predict(new)}).round(3))

1. The LPM has 41 predicted probabilities below 0 (the lowest −0.20) and none above 1. Logit and probit have none
   outside 0 to 1; their lowest are 0.017 and 0.008.
2. Very close. Membership: LPM −10.75, logit −10.84, probit −10.76 points. Web: 10.11, 9.12 and 9.08. Tenure: −2.60,
   −2.82 and −2.77 per year. Visits: −0.60, −0.70 and −0.70 per visit. Age: −0.15 in all three.
3. Customer C: 62.1% (LPM), 70.8% (logit) and 69.6% (probit). Customer D: −22.5% (LPM), 1.5% (logit) and 0.7%
   (probit). For D the LPM gives an impossible value; logit and probit agree that D is very unlikely to churn and
   differ a little in how close to 0 they put it.
4. An example: for the average difference between members and non-members, all three models give about 11 percentage
   points, so the choice does not matter for that number. We would report the logit model, because management will
   also want the risk of particular kinds of customers, and the LPM gives impossible probabilities for loyal
   customers with many visits, such as customer D. Probit gives the same conclusions, which we would mention as a
   check. (Probit with the same argument is equally good.)

## Extension tasks

### Extension 1. A retention offer for 200 customers

RetailCorp can make a retention offer to 200 customers. Take the 200 customers with the highest predicted probability
of churn in `m_logit` and count how many of them churned. How many churners would 200 customers drawn at random
contain on average? What does the comparison say about the model's use for this decision?

In [ ]:
customers["p_churn"] = m_logit.predict(customers)
top = customers.sort_values("p_churn", ascending=False).head(200)
print(f"Churners among the 200 with the highest risk: {top['churned'].sum()}")
print(f"Expected among 200 drawn at random: {200 * customers['churned'].mean():.0f}")

108 of the 200 customers with the highest predicted risk churned, against about 58 among 200 customers drawn at random.
The model does not predict well who will churn, customer by customer (Lecture 9), but it ranks customers well enough
to almost double the number of churners reached with the same number of offers. Since the model describes
associations, it does not say whether an offer would keep them; that would need a test, for example offers to a random
half of the 200.

### Extension 2. Odds ratios, and why we translate them

`np.exp(m_logit.params["member"])` is the odds ratio for membership. Compute it, and compute the odds
p / (1 − p) for customer A (Task 1, app user) and for customer B (app user, aged 45, 8 years, 60 visits), as a
non-member and as a member. What is the same for A and B, and what is not?

In [ ]:
print(f"Odds ratio for member: {np.exp(m_logit.params['member']):.3f}")
customer_b = pd.DataFrame({"member": [0, 1], "channel": "App", "tenure_years": 8, "age": 45, "visits_2025": 60})
customer_a = pd.DataFrame({"member": [0, 1], "channel": "App", "tenure_years": 1, "age": 45, "visits_2025": 15})
for name, cust in [("A", customer_a), ("B", customer_b)]:
    p = m_logit.predict(cust)
    odds = p / (1 - p)
    print(f"{name}: p {p.iloc[0]:.3f} and {p.iloc[1]:.3f}, odds {odds.iloc[0]:.3f} and {odds.iloc[1]:.3f}, "
          f"odds ratio {odds.iloc[1] / odds.iloc[0]:.3f}, gap {100 * (p.iloc[0] - p.iloc[1]):.1f} points")

The odds ratio is 0.546 for both customers: in a logit model membership multiplies the odds by the same number for
everyone, so members have 45% lower odds. The probabilities tell another story: for A the gap is 14.5 percentage points
(48.0% against 33.5%), for B 2.2 points (5.1% against 2.8%). "45% lower odds" therefore says little about how many
fewer customers churn. That is why we translate odds ratios into probabilities before we report them.